# Notebook 02: Tokenization Deep Dive

## 📚 Historical Context

**Timeline**: Evolution of Tokenization (2012-2023)

**The Problem (Pre-2012)**:
- **Word-level tokenization**: "running", "run", "runs" treated as completely different
- **Character-level**: Too many steps, loses meaning ("t", "h", "e", ...)
- **Fixed vocabulary**: Can't handle rare/new words → OOV (Out-of-Vocabulary)
- **Multilingual**: Need separate tokenizer per language

**The Evolution**:
- **2012**: BPE (Byte Pair Encoding) for data compression
- **2015**: BPE adapted for NMT (Neural Machine Translation)
- **2016**: WordPiece (Google) for speech recognition, then WMT
- **2018**: SentencePiece (Google) - language agnostic, treats text as Unicode
- **2019**: GPT-2 BPE with byte fallback - handles ANY text
- **2020**: Tokenizers library (HuggingFace) - fast Rust implementation

**Modern Impact**:
- **Subword tokenization**: Balance between words and characters
- **Open vocabulary**: Can represent any text
- **Multilingual models**: Single tokenizer for 100+ languages
- **Critical insight**: Tokenization affects model performance significantly

## 🎯 What You'll Learn

1. **BPE Algorithm** - Bottom-up merging approach
2. **WordPiece** - Likelihood-based merging
3. **SentencePiece** - Unsupervised, language-agnostic
4. **Special Tokens** - [CLS], [SEP], [PAD], [UNK], [MASK]
5. **Vocabulary Size Trade-offs** - 32k vs 50k vs 100k
6. **Tokenizer Comparison** - BERT vs GPT vs Llama
7. **Common Bugs** - And how to avoid them
8. **Custom Tokens** - Adding domain-specific vocabulary

## 💡 Why This Matters

**Tokenization is NOT just preprocessing:**
- **Wrong tokenizer** = Model never learns properly
- **Vocabulary mismatch** = Tons of [UNK] tokens
- **Poor tokenization** = Longer sequences = more memory/compute
- **Special tokens** = Critical for task performance

**Real Example**:
```python
# GPT-2 tokenizer on "Hello world!"
[15496, 995, 0]  # 3 tokens

# GPT-2 tokenizer on "안녕하세요" (Korean)
[31495, 230, 164, 123, 242, 22522, 47991, ...]  # 8+ tokens!
```

English text is over-represented in training → better tokenization.

---

In [ ]:
# Import libraries
import torch
from transformers import (
    AutoTokenizer,
    BertTokenizer,
    GPT2Tokenizer,
    LlamaTokenizer,
)
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, normalizers
from collections import Counter, defaultdict
import re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple
import json

# Set random seeds
np.random.seed(42)

# Plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

## Part 1: Understanding the Problem - Word vs Character Tokenization

### Let's see why we need subword tokenization.

In [ ]:
# Example text
text = "The runner is running faster. The fastest runners run marathons."

print("=" * 60)
print("TOKENIZATION COMPARISON")
print("=" * 60)
print(f"Original text: {text}\n")

# 1. Word-level tokenization
word_tokens = text.lower().split()
print("Word-level:")
print(f"  Tokens: {word_tokens}")
print(f"  Vocabulary: {set(word_tokens)}")
print(f"  Vocab size: {len(set(word_tokens))}")
print(f"  Problem: 'run', 'runner', 'running', 'runners' are all separate!\n")

# 2. Character-level tokenization
char_tokens = list(text.lower().replace(" ", "_"))
print("Character-level:")
print(f"  Tokens: {char_tokens[:20]}... (showing first 20)")
print(f"  Vocabulary: {set(char_tokens)}")
print(f"  Vocab size: {len(set(char_tokens))}")
print(f"  Problem: Too many tokens! Loses word boundaries!\n")

# 3. What we want: Subword tokenization
print("What we want (subword):")
print("  Tokens: ['the', 'run', '##ner', 'is', 'run', '##ning', 'fast', '##er', ...]")
print("  Benefits:")
print("    - Shares 'run' stem across variants")
print("    - Small vocabulary (30k-50k tokens)")
print("    - Can handle rare/new words by breaking them down")
print("    - Reasonable sequence length")

print("=" * 60)

## Part 2: Byte Pair Encoding (BPE) - The Foundation

### Algorithm (Bottom-Up Merging):

**1. Start with character-level vocabulary**
```
Initial: ['h', 'e', 'l', 'o', 'w', 'r', 'd']
```

**2. Count all adjacent pairs**
```
Text: "hello world"
Pairs: ('h','e'), ('e','l'), ('l','l'), ('l','o'), ('w','o'), ('o','r'), ...
```

**3. Merge most frequent pair**
```
If ('l','l') appears most: merge into 'll'
New vocab: ['h', 'e', 'll', 'o', 'w', 'r', 'd']
```

**4. Repeat until target vocabulary size**

### Key Insight:
Frequent subwords get merged first → common words become single tokens

### Used By:
- GPT-2, GPT-3, GPT-4
- RoBERTa
- BART
- Many translation models

In [ ]:
class SimpleBPE:
    """
    Simple implementation of Byte Pair Encoding for educational purposes.
    
    This is a simplified version. Production tokenizers use more optimizations.
    """
    
    def __init__(self, vocab_size=300):
        """
        Args:
            vocab_size: Target vocabulary size
        """
        self.vocab_size = vocab_size
        self.vocab = {}
        self.merges = []  # List of merge operations
    
    def get_pairs(self, word):
        """
        Get all adjacent pairs in a word.
        
        Example: 'hello' → [('h','e'), ('e','l'), ('l','l'), ('l','o')]
        """
        pairs = set()
        prev_char = word[0]
        for char in word[1:]:
            pairs.add((prev_char, char))
            prev_char = char
        return pairs
    
    def train(self, corpus: List[str]):
        """
        Train BPE on a corpus.
        
        Args:
            corpus: List of sentences/documents
        """
        # Step 1: Initialize with character-level vocabulary
        # Split each word into characters with end-of-word marker
        words = defaultdict(int)
        for text in corpus:
            for word in text.split():
                # Add space to indicate end of word
                word = ' '.join(list(word)) + ' </w>'
                words[word] += 1
        
        # Build initial vocabulary (all characters)
        vocab = set()
        for word in words:
            vocab.update(word.split())
        
        print(f"Initial vocabulary size: {len(vocab)}")
        print(f"Initial vocab: {sorted(vocab)[:20]}...\n")
        
        # Step 2: Iteratively merge most frequent pairs
        num_merges = self.vocab_size - len(vocab)
        
        for i in range(num_merges):
            # Count all pairs
            pairs = defaultdict(int)
            for word, freq in words.items():
                symbols = word.split()
                for j in range(len(symbols) - 1):
                    pairs[(symbols[j], symbols[j + 1])] += freq
            
            if not pairs:
                break
            
            # Find most frequent pair
            best_pair = max(pairs, key=pairs.get)
            
            # Merge this pair in all words
            new_words = {}
            bigram = ' '.join(best_pair)
            replacement = ''.join(best_pair)
            
            for word, freq in words.items():
                new_word = word.replace(bigram, replacement)
                new_words[new_word] = freq
            
            words = new_words
            self.merges.append(best_pair)
            vocab.add(replacement)
            
            # Print progress
            if (i + 1) % 50 == 0 or i < 5:
                print(f"Merge {i+1}: {best_pair[0]} + {best_pair[1]} → {replacement} (freq: {pairs[best_pair]})")
        
        self.vocab = vocab
        print(f"\nFinal vocabulary size: {len(self.vocab)}")
    
    def tokenize(self, text: str) -> List[str]:
        """
        Tokenize text using learned merges.
        """
        # Split into words and apply merges
        tokens = []
        for word in text.split():
            # Start with character-level
            word = ' '.join(list(word)) + ' </w>'
            
            # Apply all merges in order
            for merge in self.merges:
                bigram = ' '.join(merge)
                replacement = ''.join(merge)
                word = word.replace(bigram, replacement)
            
            tokens.extend(word.split())
        
        return tokens

# Test BPE
print("=" * 60)
print("TRAINING BPE TOKENIZER")
print("=" * 60)

# Training corpus
corpus = [
    "the runner is running",
    "faster runners run marathons",
    "the fastest marathon runners",
    "running is faster than walking",
    "the walker is walking",
] * 10  # Repeat to have enough data

bpe = SimpleBPE(vocab_size=150)
bpe.train(corpus)

# Test tokenization
print("\n" + "=" * 60)
print("TESTING BPE TOKENIZATION")
print("=" * 60)

test_sentences = [
    "the runner is running",
    "faster walking",
    "marathon",
]

for sentence in test_sentences:
    tokens = bpe.tokenize(sentence)
    print(f"\nInput:  {sentence}")
    print(f"Tokens: {tokens}")
    print(f"Count:  {len(tokens)} tokens")

print("\n" + "=" * 60)

## Part 3: WordPiece - BERT's Tokenizer

### Difference from BPE:

**BPE**: Merges most **frequent** pair

**WordPiece**: Merges pair that maximizes **likelihood** of training data

### Selection Criterion:
```
score(a, b) = freq(ab) / (freq(a) * freq(b))
```

This means:
- High score if pair appears together more than expected
- Captures stronger statistical associations

### Special Markers:
- `##` prefix for continuation tokens
- Example: "running" → ["run", "##ning"]

### Used By:
- BERT (all variants)
- DistilBERT
- ELECTRA
- Many Google models

In [ ]:
# Load BERT tokenizer to see WordPiece in action
print("=" * 60)
print("WORDPIECE TOKENIZER (BERT)")
print("=" * 60)

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

print(f"Vocabulary size: {tokenizer.vocab_size:,}")
print(f"Model max length: {tokenizer.model_max_length}\n")

# Test sentences
test_cases = [
    "running",
    "unhappiness",
    "pseudopseudohypoparathyroidism",  # Long medical term
    "Hello, how are you?",
    "PyTorch",
    "COVID-19",
]

for text in test_cases:
    # Tokenize
    tokens = tokenizer.tokenize(text)
    token_ids = tokenizer.convert_tokens_to_ids(tokens)
    
    print(f"\nText:      '{text}'")
    print(f"Tokens:    {tokens}")
    print(f"Token IDs: {token_ids}")
    print(f"Count:     {len(tokens)} tokens")
    
    # Note the ## prefix for subword continuation
    if any(token.startswith('##') for token in tokens):
        print(f"Note:      '##' indicates continuation of previous token")

print("\n" + "=" * 60)
print("KEY OBSERVATIONS")
print("=" * 60)
print("1. Common words like 'running' split into 'run' + '##ning'")
print("2. Rare words get split into many subwords")
print("3. Unknown words can still be represented (no true OOV)")
print("4. Punctuation typically gets separate tokens")
print("=" * 60)

## Part 4: SentencePiece - Modern Universal Tokenizer

### Key Innovations:

**1. Language Agnostic**
- Treats input as Unicode stream
- No language-specific preprocessing
- Works for ANY language (even mixed scripts)

**2. Reversible**
- Can perfectly reconstruct original text including spaces
- Uses ▁ (U+2581) for spaces
- Example: "Hello world" → ["▁Hello", "▁world"]

**3. Unsupervised**
- No need for pre-tokenization (word boundaries)
- Learns directly from raw text

**4. Implementations**
- Can use BPE or Unigram LM algorithm
- Unigram: Top-down (start big, prune vocabulary)

### Used By:
- T5 (all variants)
- ALBERT
- XLM-RoBERTa
- Llama / Llama 2 / Llama 3
- Mistral / Mixtral
- Most modern multilingual models

In [ ]:
# Compare different tokenizers
print("=" * 60)
print("TOKENIZER COMPARISON: BERT vs GPT-2")
print("=" * 60)

# Load different tokenizers
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
gpt2_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

# Test sentences (various languages and cases)
test_sentences = [
    "Hello, world!",
    "The quick brown fox jumps over the lazy dog.",
    "COVID-19 pandemic",
    "machine learning",
    "OpenAI GPT-3",
]

print(f"\nBERT: {bert_tokenizer.vocab_size:,} tokens (WordPiece)")
print(f"GPT-2: {gpt2_tokenizer.vocab_size:,} tokens (BPE)\n")

for text in test_sentences:
    bert_tokens = bert_tokenizer.tokenize(text)
    gpt2_tokens = gpt2_tokenizer.tokenize(text)
    
    print(f"\nText: '{text}'")
    print(f"BERT:  {bert_tokens} ({len(bert_tokens)} tokens)")
    print(f"GPT-2: {gpt2_tokens} ({len(gpt2_tokens)} tokens)")

print("\n" + "=" * 60)
print("DIFFERENCES")
print("=" * 60)
print("BERT (WordPiece):")
print("  - Lowercase only (for base model)")
print("  - Uses ## for continuation")
print("  - Separates punctuation aggressively")
print("\nGPT-2 (BPE):")
print("  - Case-sensitive")
print("  - Uses Ġ (space) prefix (shown as Ġ in output)")
print("  - Often more compact tokenization")
print("=" * 60)

## Part 5: Special Tokens - The Critical Details

### Common Special Tokens:

| Token | Purpose | Used By |
|-------|---------|----------|
| `[PAD]` | Padding sequences to same length | All models |
| `[UNK]` | Unknown/out-of-vocabulary words | All models |
| `[CLS]` | Start of sequence, used for classification | BERT |
| `[SEP]` | Separator between sentences | BERT |
| `[MASK]` | Masked token for MLM training | BERT |
| `<s>` | Start of sequence | GPT, Llama |
| `</s>` | End of sequence | GPT, Llama |
| `<|endoftext|>` | Document separator | GPT-2 |
| `<|im_start|>` | Instruction marker | ChatGPT |
| `<|im_end|>` | End instruction | ChatGPT |

### Why Special Tokens Matter:

**1. Task-specific heads**
- BERT classification: Use [CLS] token representation
- Wrong position → poor performance

**2. Attention masks**
- [PAD] tokens must be masked out
- Otherwise model learns from padding!

**3. Generation**
- Models need EOS token to know when to stop
- Missing EOS → infinite generation

**4. Multi-turn conversation**
- Special tokens mark boundaries
- Critical for chat models

In [ ]:
print("=" * 60)
print("SPECIAL TOKENS COMPARISON")
print("=" * 60)

# BERT special tokens
print("\nBERT (bert-base-uncased):")
print(f"  PAD token:   {bert_tokenizer.pad_token} (ID: {bert_tokenizer.pad_token_id})")
print(f"  UNK token:   {bert_tokenizer.unk_token} (ID: {bert_tokenizer.unk_token_id})")
print(f"  CLS token:   {bert_tokenizer.cls_token} (ID: {bert_tokenizer.cls_token_id})")
print(f"  SEP token:   {bert_tokenizer.sep_token} (ID: {bert_tokenizer.sep_token_id})")
print(f"  MASK token:  {bert_tokenizer.mask_token} (ID: {bert_tokenizer.mask_token_id})")

# GPT-2 special tokens
print("\nGPT-2 (gpt2):")
print(f"  EOS token:   {gpt2_tokenizer.eos_token} (ID: {gpt2_tokenizer.eos_token_id})")
print(f"  BOS token:   {gpt2_tokenizer.bos_token}")
print(f"  PAD token:   {gpt2_tokenizer.pad_token} (Note: GPT-2 uses EOS as PAD by default)")

# Example: BERT sequence construction
print("\n" + "=" * 60)
print("BERT SEQUENCE CONSTRUCTION")
print("=" * 60)

sentence_a = "I love machine learning"
sentence_b = "It is fascinating"

# Proper BERT format for sentence pairs
tokens = bert_tokenizer.tokenize(sentence_a)
tokens_b = bert_tokenizer.tokenize(sentence_b)

# Build sequence: [CLS] sentence_a [SEP] sentence_b [SEP]
full_sequence = [bert_tokenizer.cls_token] + tokens + [bert_tokenizer.sep_token] + tokens_b + [bert_tokenizer.sep_token]

print(f"\nSentence A: {sentence_a}")
print(f"Sentence B: {sentence_b}")
print(f"\nTokenized sequence: {full_sequence}")

# Convert to IDs
token_ids = bert_tokenizer.convert_tokens_to_ids(full_sequence)
print(f"Token IDs: {token_ids}")

# Show token type IDs (0 for sentence A, 1 for sentence B)
encoding = bert_tokenizer(
    sentence_a,
    sentence_b,
    return_tensors='pt',
    padding=True,
    truncation=True
)

print(f"\nToken type IDs: {encoding['token_type_ids'].tolist()[0]}")
print("  0 = sentence A tokens")
print("  1 = sentence B tokens")
print(f"\nAttention mask: {encoding['attention_mask'].tolist()[0]}")
print("  1 = real token, 0 = padding")

print("\n" + "=" * 60)

## Part 6: Vocabulary Size Trade-offs

### The Dilemma:

**Small Vocabulary (e.g., 8k tokens)**
- ✓ Smaller embedding layer → less memory
- ✓ Faster training (fewer tokens to learn)
- ✗ Longer sequences (words split more)
- ✗ More computation (quadratic attention)

**Large Vocabulary (e.g., 100k tokens)**
- ✓ Shorter sequences (whole words)
- ✓ Better rare word representation
- ✗ Huge embedding layer → more memory
- ✗ Many rare tokens → hard to learn

### Common Configurations:

| Model | Vocabulary Size | Reasoning |
|-------|----------------|----------|
| BERT | 30,522 | Balanced for English |
| GPT-2 | 50,257 | Larger for better generation |
| GPT-3 | 50,257 | Same as GPT-2 |
| T5 | 32,000 | SentencePiece, multilingual |
| Llama | 32,000 | Efficient multilingual |
| Llama 2 | 32,000 | Same as Llama |
| GPT-4 | 100,277 | Much larger for quality |

### Rule of Thumb:
- **30k-50k**: Good default for most applications
- **Multilingual**: Need larger vocab (100k+)
- **Code**: Need larger vocab (special characters, indentation)

In [ ]:
print("=" * 60)
print("VOCABULARY SIZE IMPACT")
print("=" * 60)

# Test text
test_text = """The quick brown fox jumps over the lazy dog. 
This pangram contains every letter of the alphabet. 
Machine learning models use tokenization to process text efficiently."""

# Different tokenizers with different vocab sizes
tokenizers_to_test = [
    ('bert-base-uncased', 'BERT'),
    ('gpt2', 'GPT-2'),
    ('t5-small', 'T5'),
]

results = []

for model_name, display_name in tokenizers_to_test:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokens = tokenizer.tokenize(test_text)
    
    results.append({
        'name': display_name,
        'vocab_size': tokenizer.vocab_size,
        'num_tokens': len(tokens),
        'tokens': tokens[:10]  # First 10 for display
    })
    
    print(f"\n{display_name}:")
    print(f"  Vocabulary size: {tokenizer.vocab_size:,}")
    print(f"  Text length:     {len(test_text)} characters")
    print(f"  Number of tokens: {len(tokens)}")
    print(f"  Tokens/char ratio: {len(tokens)/len(test_text):.3f}")
    print(f"  First 10 tokens: {tokens[:10]}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Vocab size vs number of tokens
names = [r['name'] for r in results]
vocab_sizes = [r['vocab_size'] for r in results]
num_tokens = [r['num_tokens'] for r in results]

axes[0].bar(names, vocab_sizes, color='steelblue', alpha=0.7)
axes[0].set_ylabel('Vocabulary Size')
axes[0].set_title('Tokenizer Vocabulary Sizes')
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(names, num_tokens, color='coral', alpha=0.7)
axes[1].set_ylabel('Number of Tokens')
axes[1].set_title('Tokens Generated for Same Text')
axes[1].grid(axis='y', alpha=0.3)
axes[1].axhline(y=min(num_tokens), color='red', linestyle='--', alpha=0.5, label='Minimum')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("KEY INSIGHT")
print("=" * 60)
print("Larger vocabulary does NOT always mean fewer tokens!")
print("It depends on:")
print("  1. Training data distribution")
print("  2. Tokenization algorithm (BPE vs WordPiece vs SentencePiece)")
print("  3. Language and domain coverage")
print("=" * 60)

## Part 7: Common Tokenization Bugs and How to Fix Them

### Bug #1: Tokenizer Mismatch
**Problem**: Training with one tokenizer, inference with another
```python
# WRONG
model = BertModel.from_pretrained('bert-base-uncased')
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')  # WRONG!
```

**Fix**: Always use matching tokenizer
```python
# CORRECT
model = BertModel.from_pretrained('bert-base-uncased')
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
```

### Bug #2: Missing Special Tokens
**Problem**: Not adding [CLS], [SEP] for BERT classification
```python
# WRONG - missing special tokens
tokens = tokenizer.tokenize("Hello world")
```

**Fix**: Use proper encoding
```python
# CORRECT
encoding = tokenizer("Hello world", return_tensors='pt')
```

### Bug #3: Ignoring Attention Masks
**Problem**: Not masking padding tokens
```python
# WRONG - model learns from padding!
outputs = model(input_ids)  # No attention mask
```

**Fix**: Always pass attention mask
```python
# CORRECT
outputs = model(input_ids, attention_mask=attention_mask)
```

### Bug #4: Exceeding Max Length
**Problem**: Sequences longer than model's max length

**Fix**: Truncate or use sliding window
```python
encoding = tokenizer(
    text,
    truncation=True,
    max_length=512,
    return_tensors='pt'
)
```

### Bug #5: Not Setting Padding Side for Generation
**Problem**: GPT models need padding on LEFT for generation

**Fix**: Set padding side
```python
tokenizer.padding_side = 'left'  # For GPT generation
```

In [ ]:
print("=" * 60)
print("COMMON TOKENIZATION BUGS DEMONSTRATION")
print("=" * 60)

# Bug #1: Tokenizer Mismatch
print("\n1. TOKENIZER MISMATCH")
print("-" * 40)
text = "Hello, how are you?"

bert_tok = BertTokenizer.from_pretrained('bert-base-uncased')
gpt2_tok = GPT2Tokenizer.from_pretrained('gpt2')

bert_tokens = bert_tok.tokenize(text)
gpt2_tokens = gpt2_tok.tokenize(text)

print(f"Text: '{text}'")
print(f"BERT tokens:  {bert_tokens}")
print(f"GPT-2 tokens: {gpt2_tokens}")
print("⚠️  These are DIFFERENT! Using wrong tokenizer = broken model")

# Bug #2: Missing Special Tokens
print("\n2. MISSING SPECIAL TOKENS")
print("-" * 40)

# Wrong way
tokens_wrong = bert_tok.tokenize(text)
ids_wrong = bert_tok.convert_tokens_to_ids(tokens_wrong)

# Correct way
encoding_correct = bert_tok(text, return_tensors='pt')
ids_correct = encoding_correct['input_ids'][0].tolist()
tokens_correct = bert_tok.convert_ids_to_tokens(ids_correct)

print(f"❌ Wrong (manual tokenization): {tokens_wrong}")
print(f"✓ Correct (using tokenizer):   {tokens_correct}")
print("Notice: Correct version has [CLS] at start and [SEP] at end")

# Bug #3: Ignoring Attention Masks
print("\n3. ATTENTION MASKS")
print("-" * 40)

# Batch of sentences with different lengths
sentences = [
    "Short sentence",
    "This is a much longer sentence with more words"
]

# Encode with padding
encoding = bert_tok(sentences, padding=True, return_tensors='pt')

print("Input IDs shape:", encoding['input_ids'].shape)
print("\nAttention masks:")
for i, mask in enumerate(encoding['attention_mask']):
    print(f"  Sentence {i+1}: {mask.tolist()}")
    print(f"              1 = real token, 0 = padding")

print("\n⚠️  Always pass attention_mask to model!")

# Bug #4: Exceeding Max Length
print("\n4. EXCEEDING MAX LENGTH")
print("-" * 40)

long_text = "word " * 1000  # Very long text

# Without truncation (will fail or truncate silently)
try:
    encoding_no_trunc = bert_tok(long_text, return_tensors='pt')
    print(f"❌ No truncation: {encoding_no_trunc['input_ids'].shape[1]} tokens")
    print(f"   (Model max: {bert_tok.model_max_length})")
except Exception as e:
    print(f"❌ Error: {e}")

# With truncation
encoding_trunc = bert_tok(
    long_text,
    truncation=True,
    max_length=512,
    return_tensors='pt'
)
print(f"✓ With truncation: {encoding_trunc['input_ids'].shape[1]} tokens")
print(f"  Safely truncated to model's max length")

# Bug #5: Padding Side
print("\n5. PADDING SIDE FOR GENERATION")
print("-" * 40)

# For GPT models, padding should be on left
gpt2_tok.pad_token = gpt2_tok.eos_token  # GPT-2 doesn't have pad token by default

print("Default padding side:", gpt2_tok.padding_side)

# Set to left for generation
gpt2_tok.padding_side = 'left'
print("Changed to:", gpt2_tok.padding_side)
print("\nWhy? For generation, padding must be on LEFT")
print("Otherwise, model generates from padding instead of text!")

print("\n" + "=" * 60)

## Part 8: Adding Custom Tokens

### When to Add Custom Tokens:

**1. Domain-specific terms**
- Medical: drug names, diseases
- Legal: case citations, legal terms
- Technical: product names, abbreviations

**2. Frequently occurring patterns**
- Email addresses: user@domain.com
- URLs: https://example.com
- Hashtags: #MachineLearning

**3. Task-specific markers**
- Instruction tuning: `<INST>`, `</INST>`
- Multi-turn chat: `<USER>`, `<ASSISTANT>`

### How to Add:

**1. Add to tokenizer**
```python
tokenizer.add_tokens(['<CUSTOM>', '<MARKER>'])
```

**2. Resize model embeddings**
```python
model.resize_token_embeddings(len(tokenizer))
```

**3. CRITICAL: Initialize new embeddings**
- Random: Model will learn from scratch
- Mean of existing: Better initialization
- Don't forget this step!

### Trade-offs:
- ✓ Better representation of domain terms
- ✓ Shorter sequences for common patterns
- ✗ More parameters (embedding layer grows)
- ✗ Need to train the new embeddings

In [ ]:
from transformers import AutoModel

print("=" * 60)
print("ADDING CUSTOM TOKENS")
print("=" * 60)

# Load a small model and tokenizer
model_name = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

print(f"\nOriginal vocabulary size: {len(tokenizer)}")
print(f"Model embedding size: {model.embeddings.word_embeddings.weight.shape}")

# Define custom tokens
custom_tokens = [
    '<MEDICAL>',
    '<LEGAL>',
    '<TECHNICAL>',
    '[COVID-19]',
    '[PyTorch]',
]

print(f"\nAdding {len(custom_tokens)} custom tokens: {custom_tokens}")

# Add tokens to tokenizer
num_added = tokenizer.add_tokens(custom_tokens)
print(f"Successfully added: {num_added} tokens")

# Resize model embeddings
print("\nResizing model embeddings...")
model.resize_token_embeddings(len(tokenizer))

print(f"New vocabulary size: {len(tokenizer)}")
print(f"New embedding size: {model.embeddings.word_embeddings.weight.shape}")

# Test the new tokens
print("\n" + "=" * 60)
print("TESTING CUSTOM TOKENS")
print("=" * 60)

test_text = "<MEDICAL> This patient has [COVID-19] and uses [PyTorch] for research."

# Tokenize
tokens = tokenizer.tokenize(test_text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print(f"\nText: {test_text}")
print(f"\nTokens: {tokens}")
print(f"\nToken IDs: {token_ids}")

# Show which tokens are custom (have high IDs)
print(f"\nCustom token IDs (>= {tokenizer.vocab_size - num_added}):")
for token, token_id in zip(tokens, token_ids):
    if token_id >= tokenizer.vocab_size - num_added:
        print(f"  '{token}' → ID {token_id} ✓ (custom)")

# IMPORTANT: Initialize new embeddings properly
print("\n" + "=" * 60)
print("INITIALIZING NEW EMBEDDINGS")
print("=" * 60)

with torch.no_grad():
    # Get embeddings
    embeddings = model.embeddings.word_embeddings.weight
    
    # Get mean of existing embeddings
    mean_embedding = embeddings[:tokenizer.vocab_size - num_added].mean(dim=0)
    std_embedding = embeddings[:tokenizer.vocab_size - num_added].std(dim=0)
    
    # Initialize new tokens with similar statistics
    for i in range(num_added):
        idx = tokenizer.vocab_size - num_added + i
        embeddings[idx] = mean_embedding + torch.randn(embeddings.shape[1]) * std_embedding * 0.01

print("✓ New embeddings initialized with mean + small random noise")
print("  This helps the model learn them faster than random initialization")

print("\n" + "=" * 60)
print("BEST PRACTICES")
print("=" * 60)
print("1. Add custom tokens BEFORE starting training")
print("2. Always resize model embeddings after adding tokens")
print("3. Initialize new embeddings (mean of existing + noise)")
print("4. Save both tokenizer and model together")
print("5. Don't add too many tokens (increases model size)")
print("=" * 60)

## Part 9: Visualizing Tokenization - Understanding the Splits

In [ ]:
import pandas as pd

print("=" * 60)
print("TOKENIZATION VISUALIZATION")
print("=" * 60)

# Test various text types
test_texts = [
    "Hello world!",
    "COVID-19 pandemic",
    "user@example.com",
    "https://www.example.com",
    "#MachineLearning",
    "anti-inflammatory",
    "supercalifragilisticexpialidocious",
    "I'm running quickly",
    "PyTorch 2.0 is fast",
]

# Compare tokenizers
bert_tok = BertTokenizer.from_pretrained('bert-base-uncased')
gpt2_tok = GPT2Tokenizer.from_pretrained('gpt2')

# Analyze
results = []
for text in test_texts:
    bert_tokens = bert_tok.tokenize(text)
    gpt2_tokens = gpt2_tok.tokenize(text)
    
    results.append({
        'Text': text,
        'Chars': len(text),
        'BERT Tokens': len(bert_tokens),
        'GPT-2 Tokens': len(gpt2_tokens),
        'BERT Split': ' | '.join(bert_tokens),
        'GPT-2 Split': ' | '.join(gpt2_tokens),
    })

df = pd.DataFrame(results)
print("\n", df.to_string(index=False))

# Visualize token counts
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(test_texts))
width = 0.35

ax.bar(x - width/2, df['BERT Tokens'], width, label='BERT', alpha=0.8, color='steelblue')
ax.bar(x + width/2, df['GPT-2 Tokens'], width, label='GPT-2', alpha=0.8, color='coral')

ax.set_xlabel('Test Text')
ax.set_ylabel('Number of Tokens')
ax.set_title('Tokenization Comparison: BERT vs GPT-2')
ax.set_xticks(x)
ax.set_xticklabels([t[:20] + '...' if len(t) > 20 else t for t in test_texts], rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("KEY OBSERVATIONS")
print("=" * 60)
print("1. Special characters (URLs, emails) tokenize poorly")
print("2. Rare/long words get split into many pieces")
print("3. Different tokenizers handle same text differently")
print("4. This affects: sequence length, attention cost, model performance")
print("=" * 60)

## 📊 Summary and Key Takeaways

### What We Learned:

**1. Tokenization Algorithms**
- **BPE**: Bottom-up, frequency-based merging (GPT-2, GPT-3)
- **WordPiece**: Likelihood-based merging (BERT)
- **SentencePiece**: Language-agnostic, reversible (T5, Llama)

**2. Special Tokens**
- [PAD], [UNK], [CLS], [SEP], [MASK] for BERT
- <s>, </s>, <|endoftext|> for GPT
- Critical for task-specific performance

**3. Vocabulary Size**
- Trade-off: size vs sequence length vs rare words
- Common: 30k-50k tokens
- Multilingual/Code: 100k+ tokens

**4. Common Bugs**
- Tokenizer mismatch (different model/tokenizer)
- Missing special tokens
- Ignoring attention masks
- Exceeding max length
- Wrong padding side for generation

**5. Custom Tokens**
- Add domain-specific terms
- Resize embeddings
- Initialize properly (mean + noise)

### Critical for Fine-Tuning:

**Before Training**:
1. ✓ Use correct tokenizer for your base model
2. ✓ Add custom tokens if needed
3. ✓ Test tokenization on your data
4. ✓ Check sequence length distribution
5. ✓ Verify special tokens are correct

**During Training**:
1. ✓ Always pass attention masks
2. ✓ Use proper truncation/padding
3. ✓ Monitor [UNK] token frequency
4. ✓ Watch for OOV issues

**Common Gotchas**:
```python
# ❌ DON'T
tokens = tokenizer.tokenize(text)  # Missing special tokens
model(input_ids)  # Missing attention mask

# ✓ DO
encoding = tokenizer(text, return_tensors='pt', truncation=True)
model(**encoding)  # Unpacks all needed tensors
```

### Real-World Impact:

| Issue | Impact | Solution |
|-------|--------|----------|
| Wrong tokenizer | Model fails to learn | Use matching tokenizer |
| Missing [CLS] | Poor classification | Use tokenizer() method |
| No attention mask | Learns from padding | Always pass mask |
| Long sequences | OOM errors | Truncate or use sliding window |
| High [UNK] rate | Poor performance | Add custom tokens or retrain tokenizer |

### Next Notebook:
**03_dataset_preparation_basics.ipynb**

We'll explore:
- Loading data from various formats
- Dataset formats for different tasks
- Train/val/test splitting strategies
- Data streaming for large files
- Building preprocessing pipelines

---

**Historical Note**: Before subword tokenization (pre-2015), NMT models struggled with rare words and morphologically rich languages. BPE's introduction in 2015 was a breakthrough moment - suddenly models could handle any word by breaking it into subwords. This single innovation enabled the multilingual models we have today (100+ languages in one model). The tokenization algorithm you choose still matters significantly for model performance, especially in specialized domains.